# השוואת כלים עבור 32 מועמדי HSV-2

המחברת טוענת את תוצרי v0.3 ומפעילה את שכבת v0.4. שלושים ושניים המועמדים הם מועמדים חישוביים שעבורם Cas-OFFinder לא מצא פגיעה אנושית עד שלושה mismatches במודל SpCas9 שהוגדר. זו אינה הוכחת בטיחות, יעילות או תיקוף.

In [ ]:
from pathlib import Path
import pandas as pd
import viral_safe_target as vst

run = vst.load_run(Path('../reports/hsv2_pilot'))
candidates = run.candidates.query('human_total_predicted_hits == 0').copy()
assert len(candidates) == 32
candidates['gene_name'].value_counts()

## מה כל כלי מוסיף?

ViralSafeTarget מדרג שימור ותכונות רצף; Cas-OFFinder מחפש התאמות ברפרנס; CRISPRitz יכול להוסיף bulges וגרסאות; CRISPOR, CHOPCHOP ו-GuideScan2 נקלטים מיצוא מתועד. ציונים גולמיים מודדים דברים שונים ולכן משווים ranks ואחוזונים בתוך כל כלי, לא ממוצע גולמי.

In [ ]:
baseline = vst.candidate_metrics_as_tool_results(candidates)
comparison = vst.compare_tools(
    candidates, [baseline],
    expected_tools=['viral_safe_target_pre_human', 'viral_safe_target_post_human',
                    'cas-offinder', 'crispritz', 'crispor', 'chopchop', 'guidescan2'],
)
comparison.candidate_tool_matrix.head()

## מועמדים מובילים וחוסר הסכמה

המטריצה שומרת NaN כאשר כלי חסר. `tools_missing`, שונות הדירוג ו-`disagreement_score` מונעים מהקונצנזוס להסתיר כיסוי חלקי.

In [ ]:
display(comparison.consensus_candidates.head(10))
display(comparison.disagreement_report.head(10))
display(comparison.tool_coverage)

## scorer מותאם אישית

חוקר יכול לממש אובייקט עם `name`, `version` ומתודת `score(candidates)`. הדוגמה השקופה אינה מודל ביולוגי מאומת.

In [ ]:
scored = vst.ExampleRuleScorer().score(candidates)
scored.head()

## תוצאות CRISPResso2 עתידיות

CRISPResso2 מייצג מדידות מריצת sequencing קיימת. היבוא נשמר תחת `experimental/` ואינו נכנס אוטומטית לציוני החיזוי. השוואה עתידית תציג תחזית מול מדידה, בלי להפוך מדידה יחידה להוכחת בטיחות או יעילות.